# WGAN-GP Implementation on CelebA

This notebook implements Wasserstein GAN with Gradient Penalty (WGAN-GP) for the CelebA dataset.

## Key Concepts:
- **Wasserstein Distance**: Provides more stable training than JS divergence
- **Gradient Penalty**: Enforces 1-Lipschitz constraint without weight clipping
- **NO BatchNorm in Critic**: BatchNorm normalizes activations, interfering with Lipschitz constraint
- **N-Critic Training**: Train critic 5x per generator update

## Setup and Imports

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install matplotlib numpy pillow -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchvision.utils import make_grid, save_image
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
from tqdm import tqdm

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Part 1: Define Generator Architecture

In [ ]:
class Generator(nn.Module):
    """
    Generator Architecture for WGAN-GP
    
    Input: Latent vector z (batch_size, 128)
    Output: Generated image (batch_size, 3, 64, 64) in [-1, 1]
    
    Architecture:
    - FC layer: 128 -> 256*4*4
    - 4x ConvTranspose2d blocks with BatchNorm + ReLU
    - Final Tanh to ensure output in [-1, 1]
    """
    def __init__(self, latent_dim=128, img_channels=3):
        super(Generator, self).__init__()
        self.latent_dim = latent_dim
        
        # FC layer projects latent vector to spatial feature map
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4)
        
        # Progressive upsampling via ConvTranspose2d
        self.blocks = nn.Sequential(
            # 256x4x4 -> 128x8x8
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            
            # 128x8x8 -> 64x16x16
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            
            # 64x16x16 -> 32x32x32
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            
            # 32x32x32 -> 3x64x64
            nn.ConvTranspose2d(32, img_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()  # Output in [-1, 1]
        )
    
    def forward(self, z):
        # Project and reshape
        x = self.fc(z)
        x = x.view(-1, 256, 4, 4)
        # Upsample through blocks
        x = self.blocks(x)
        return x

# Test generator
gen = Generator().to(device)
z_test = torch.randn(4, 128).to(device)
img_test = gen(z_test)
print(f"Generator output shape: {img_test.shape}")
print(f"Output range: [{img_test.min():.3f}, {img_test.max():.3f}]")

## Part 2: Define Critic Architecture (NO BatchNorm!)

In [ ]:
class Critic(nn.Module):
    """
    Critic (Discriminator) for WGAN-GP
    
    ⚠️ CRITICAL: NO BatchNorm in Critic ⚠️
    
    Why?
    - BatchNorm normalizes activations to mean=0, std=1
    - This breaks the Lipschitz constraint that WGAN relies on
    - The gradient penalty needs to work on the true input distribution
    - If we normalize, the gradient penalty becomes meaningless
    
    What we use instead:
    - LeakyReLU for negative slope (prevents dying ReLU)
    - Conv2d with bias=True (since we can't use BatchNorm)
    - Stride-2 convolutions for downsampling
    
    Input: Image (batch_size, 3, 64, 64)
    Output: Unbounded scalar (batch_size, 1)
    """
    def __init__(self, img_channels=3):
        super(Critic, self).__init__()
        
        # Progressive downsampling
        self.blocks = nn.Sequential(
            # 3x64x64 -> 32x32x32
            nn.Conv2d(img_channels, 32, kernel_size=4, stride=2, padding=1, bias=True),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 32x32x32 -> 64x16x16
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1, bias=True),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 64x16x16 -> 128x8x8
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=True),
            nn.LeakyReLU(0.2, inplace=True),
            
            # 128x8x8 -> 256x4x4
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=True),
            nn.LeakyReLU(0.2, inplace=True),
        )
        
        # Output layer: 256x4x4 -> 1 (unbounded scalar)
        # NO sigmoid or activation - we want unbounded output
        self.fc = nn.Linear(256 * 4 * 4, 1, bias=True)
    
    def forward(self, img):
        x = self.blocks(img)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Test critic
crit = Critic().to(device)
img_test = torch.randn(4, 3, 64, 64).to(device)
score = crit(img_test)
print(f"Critic output shape: {score.shape}")
print(f"Score range: [{score.min():.3f}, {score.max():.3f}] (unbounded - good!)")

## Part 3: Gradient Penalty Implementation

In [ ]:
def compute_gradient_penalty(critic, real_images, fake_images, device, lambda_gp=10):
    """
    Compute Gradient Penalty
    
    The gradient penalty enforces the 1-Lipschitz constraint:
    ||∇_x D(x)|| ≤ 1 for all x
    
    We enforce this by:
    1. Interpolating between real and fake samples: x_hat = α*x_real + (1-α)*x_fake
    2. Computing gradient: ∇_x_hat D(x_hat)
    3. Penalizing deviation from unit norm: (||∇_x_hat D(x_hat)|| - 1)^2
    
    Args:
        critic: Critic network
        real_images: Batch of real images
        fake_images: Batch of fake images
        device: torch device
        lambda_gp: Penalty coefficient (default: 10)
    
    Returns:
        Gradient penalty loss (scalar)
    """
    batch_size = real_images.size(0)
    
    # Step 1: Sample random interpolation coefficient
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    
    # Step 2: Interpolate between real and fake
    interpolates = (alpha * real_images + (1 - alpha) * fake_images).requires_grad_(True)
    
    # Step 3: Get critic scores for interpolated samples
    d_interpolates = critic(interpolates)
    
    # Step 4: Compute gradients
    fake_labels = torch.ones(batch_size, 1, device=device)
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake_labels,
        create_graph=True,
        retain_graph=True,
    )[0]
    
    # Step 5: Compute gradient norms
    gradients = gradients.view(batch_size, -1)
    gradient_norms = torch.sqrt(torch.sum(gradients ** 2, dim=1) + 1e-8)
    
    # Step 6: Penalize deviation from unit norm
    gradient_penalty = lambda_gp * torch.mean((gradient_norms - 1) ** 2)
    
    return gradient_penalty

print("Gradient penalty function defined!")

## Part 4: Data Loading

In [ ]:
def get_celeba_dataloader(batch_size=64, image_size=64, num_workers=2, root="./data"):
    """
    Load CelebA dataset with preprocessing
    
    Key preprocessing steps:
    - Resize to 64x64
    - Center crop to remove borders
    - Convert to tensor
    - Normalize to [-1, 1] (important for Tanh activation!)
    """
    # Define preprocessing pipeline
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        # Normalize: (x - 0.5) / 0.5 = 2x - 1  (maps [0,1] to [-1,1])
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    
    # Load CelebA
    dataset = datasets.CelebA(
        root=root,
        split='train',
        transform=transform,
        download=True  # Will download if not present
    )
    
    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True  # Drop incomplete batch
    )
    
    return dataloader

print("Loading CelebA dataset...")
# Note: This may take a while on first run as it downloads the dataset (~200GB)
# For faster testing, you can reduce batch_size or use fewer samples
dataloader = get_celeba_dataloader(batch_size=32, num_workers=2)

# Check one batch
real_batch, _ = next(iter(dataloader))
print(f"Real image batch shape: {real_batch.shape}")
print(f"Pixel range: [{real_batch.min():.3f}, {real_batch.max():.3f}]")
print(f"Dataset loaded successfully!")

## Part 5: Initialize Models and Optimizers

In [ ]:
# Hyperparameters
LATENT_DIM = 128
LR = 1e-4
BETA1 = 0.0  # ← Important for WGAN! Zero out momentum
BETA2 = 0.9
LAMBDA_GP = 10
N_CRITIC = 5  # Train critic 5x per generator update

# Initialize networks
generator = Generator(latent_dim=LATENT_DIM).to(device)
critic = Critic().to(device)

# Initialize optimizers with WGAN-specific settings
optimizer_g = optim.Adam(generator.parameters(), lr=LR, betas=(BETA1, BETA2))
optimizer_c = optim.Adam(critic.parameters(), lr=LR, betas=(BETA1, BETA2))

# Count parameters
n_params_g = sum(p.numel() for p in generator.parameters())
n_params_c = sum(p.numel() for p in critic.parameters())

print(f"Generator parameters: {n_params_g:,}")
print(f"Critic parameters: {n_params_c:,}")
print(f"Total parameters: {n_params_g + n_params_c:,}")
print(f"\nOptimizer settings:")
print(f"  LR: {LR}")
print(f"  Beta1 (momentum): {BETA1}")
print(f"  Beta2 (RMSprop): {BETA2}")
print(f"  Gradient Penalty Lambda: {LAMBDA_GP}")
print(f"  N-Critic: {N_CRITIC}")

## Part 6: Training Loop

In [ ]:
# Storage for metrics
losses_g = []
losses_c = []
losses_gp = []
wasserstein_distances = []

# Training configuration
NUM_EPOCHS = 50  # Adjust based on compute budget
SAVE_INTERVAL = 10
PLOT_INTERVAL = 100

print(f"Starting training for {NUM_EPOCHS} epochs...\n")

for epoch in range(NUM_EPOCHS):
    for batch_idx, (real_images, _) in enumerate(dataloader):
        real_images = real_images.to(device)
        batch_size = real_images.size(0)
        
        # ===== CRITIC UPDATE =====
        for critic_iter in range(N_CRITIC):
            optimizer_c.zero_grad()
            
            # Sample random latent vectors
            z = torch.randn(batch_size, LATENT_DIM, device=device)
            
            # Generate fake images (detach to avoid updating generator)
            fake_images = generator(z).detach()
            
            # Get critic scores
            real_scores = critic(real_images)
            fake_scores = critic(fake_images)
            
            # Wasserstein distance: E[D(real)] - E[D(fake)]
            wasserstein_dist = real_scores.mean() - fake_scores.mean()
            
            # Gradient penalty
            gp = compute_gradient_penalty(critic, real_images, fake_images, device, LAMBDA_GP)
            
            # Critic loss (minimize negative Wasserstein + penalty)
            critic_loss = -wasserstein_dist + gp
            
            critic_loss.backward()
            optimizer_c.step()
        
        # ===== GENERATOR UPDATE =====
        optimizer_g.zero_grad()
        
        # Sample new latent vectors
        z = torch.randn(batch_size, LATENT_DIM, device=device)
        
        # Generate fake images
        fake_images = generator(z)
        
        # Get critic scores
        fake_scores = critic(fake_images)
        
        # Generator loss (maximize fake scores)
        generator_loss = -fake_scores.mean()
        
        generator_loss.backward()
        optimizer_g.step()
        
        # Store metrics
        losses_g.append(generator_loss.item())
        losses_c.append(critic_loss.item())
        losses_gp.append(gp.item())
        wasserstein_distances.append(wasserstein_dist.item())
        
        # Print progress
        if (batch_idx + 1) % PLOT_INTERVAL == 0:
            print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Batch {batch_idx+1}/{len(dataloader)} | "
                  f"G_loss: {generator_loss.item():.4f} | "
                  f"C_loss: {critic_loss.item():.4f} | "
                  f"GP: {gp.item():.4f} | "
                  f"W_dist: {wasserstein_dist.item():.4f}")
    
    # Save checkpoint periodically
    if (epoch + 1) % SAVE_INTERVAL == 0:
        checkpoint = {
            'epoch': epoch + 1,
            'generator': generator.state_dict(),
            'critic': critic.state_dict(),
            'optimizer_g': optimizer_g.state_dict(),
            'optimizer_c': optimizer_c.state_dict(),
        }
        torch.save(checkpoint, f'checkpoint_epoch_{epoch+1}.pt')
        print(f"\nCheckpoint saved at epoch {epoch+1}\n")

print("Training completed!")

## Part 7: Visualization - Loss Curves

In [ ]:
# Plot training losses
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Generator loss
axes[0, 0].plot(losses_g, label='Generator Loss', color='blue')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Generator Loss Over Time')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# Critic loss
axes[0, 1].plot(losses_c, label='Critic Loss', color='orange')
axes[0, 1].set_xlabel('Iteration')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Critic Loss Over Time')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Gradient penalty
axes[1, 0].plot(losses_gp, label='Gradient Penalty', color='green')
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Gradient Penalty Over Time')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Wasserstein distance
axes[1, 1].plot(wasserstein_distances, label='Wasserstein Distance', color='red')
axes[1, 1].set_xlabel('Iteration')
axes[1, 1].set_ylabel('Distance')
axes[1, 1].set_title('Wasserstein Distance (should decrease)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training curves saved to 'training_curves.png'")

## Part 8: Generate 50 Samples in a Grid

In [ ]:
def denormalize(images):
    """Convert from [-1, 1] to [0, 1] for visualization"""
    return (images + 1) / 2

# Generate 50 samples
num_samples = 50
generator.eval()

with torch.no_grad():
    z = torch.randn(num_samples, LATENT_DIM, device=device)
    generated_images = generator(z)
    generated_images = denormalize(generated_images)
    generated_images = torch.clamp(generated_images, 0, 1)

# Create grid (5x10)
grid = make_grid(generated_images, nrow=10, normalize=False, padding=2)

# Display
fig, ax = plt.subplots(figsize=(20, 10))
ax.imshow(grid.permute(1, 2, 0).cpu().numpy())
ax.axis('off')
ax.set_title('Generated Samples (5x10 Grid)', fontsize=16)
plt.tight_layout()
plt.savefig('generated_samples_grid.png', dpi=150, bbox_inches='tight')
plt.show()

# Save as raw image
save_image(generated_images, 'generated_samples.png', nrow=10)
print(f"Generated 50 samples and saved to 'generated_samples.png'")

## Part 9: Latent Interpolation (8 steps)

In [ ]:
# Latent interpolation between two random vectors
num_steps = 8
num_interpolations = 5  # Show 5 different interpolations

generator.eval()

all_interpolations = []

with torch.no_grad():
    for interp_idx in range(num_interpolations):
        # Sample two random latent vectors
        z1 = torch.randn(1, LATENT_DIM, device=device)
        z2 = torch.randn(1, LATENT_DIM, device=device)
        
        # Linear interpolation: z(t) = (1-t)*z1 + t*z2
        for t in torch.linspace(0, 1, num_steps):
            z_interp = (1 - t) * z1 + t * z2
            img = generator(z_interp)
            img = denormalize(img)
            img = torch.clamp(img, 0, 1)
            all_interpolations.append(img)

# Concatenate all interpolations
all_interpolations = torch.cat(all_interpolations, dim=0)

# Create grid (5 rows x 8 columns = 5x8)
grid_interp = make_grid(all_interpolations, nrow=num_steps, normalize=False, padding=2)

# Display
fig, ax = plt.subplots(figsize=(16, 10))
ax.imshow(grid_interp.permute(1, 2, 0).cpu().numpy())
ax.axis('off')
ax.set_title('Latent Interpolation (5 interpolations x 8 steps each)', fontsize=16)
plt.tight_layout()
plt.savefig('latent_interpolation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Generated latent interpolations and saved to 'latent_interpolation.png'")

## Analysis: WGAN-GP vs Original GAN

### Original GAN (Goodfellow et al., 2014):
- **Loss:** Jensen-Shannon divergence
- **Discriminator:** Binary classifier with sigmoid output
- **Issues:** Mode collapse, training instability, vanishing gradients

### WGAN-GP (Improved Wasserstein GAN with Gradient Penalty):
- **Loss:** Wasserstein distance (earth-mover distance)
- **Critic:** Unbounded output (no activation function)
- **Improvements:**
  - More stable training (no vanishing gradients)
  - Better convergence
  - Meaningful loss values that correlate with image quality
  - Gradient penalty enforces 1-Lipschitz constraint without weight clipping

### Key Differences:
1. **Distance Metric:** JS divergence → Wasserstein distance
2. **Output Structure:** Binary classifier → Unbounded critic
3. **Normalization:** No constraints → Gradient penalty
4. **Training Dynamics:** Unstable → More stable

In [ ]:
# Summary statistics
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Total iterations: {len(losses_g)}")
print(f"Total epochs: {NUM_EPOCHS}")
print(f"\nFinal metrics:")
print(f"  Generator Loss: {losses_g[-1]:.4f}")
print(f"  Critic Loss: {losses_c[-1]:.4f}")
print(f"  Gradient Penalty: {losses_gp[-1]:.4f}")
print(f"  Wasserstein Distance: {wasserstein_distances[-1]:.4f}")
print(f"\nAverages (last 100 iterations):")
print(f"  Avg G Loss: {np.mean(losses_g[-100:]):.4f}")
print(f"  Avg C Loss: {np.mean(losses_c[-100:]):.4f}")
print(f"  Avg GP: {np.mean(losses_gp[-100:]):.4f}")
print(f"="*60)